# Lesson 7 exercise — QLoRA on a free GPU

**Goal:** fine-tune `SmolLM2-1.7B-Instruct`, a model whose full fine-tune would need about twice a T4's memory, by loading it in 4-bit and training LoRA adapters on top. Along the way, predict the memory with arithmetic and then measure it.

**How to use this notebook**
1. In Colab: *Runtime → Change runtime type → T4 GPU*. This notebook needs the GPU.
2. Run cells top to bottom. Wherever you see `...` after a `# TODO`, replace it with your code.
3. Run the **check** cell under each TODO. ✅ means you've got it; ❌ gives you a hint.
4. The model download is about 3.4 GB, so the loading cell takes a minute or two.
5. Want to re-run training with different settings? Start again from **section 3**. It reloads a clean model, because the trainer adds the adapters to `model` in place.

Stuck? Look up the error in the course's troubleshooting sheet (`reference/troubleshooting.html`). Still stuck? Ask on the Hugging Face Forums with your code and the full traceback.

## 0 · Setup, task data and metric
The usual three cells, plus `bitsandbytes`, the library that stores weights in 4-bit.

In [ ]:
%pip install -q -U "transformers>=5.19,<6" "trl>=1.15,<2" "peft>=0.21,<1" "datasets>=5,<6" "accelerate>=1.15,<2"
import torch, transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

def check(passed, ok_msg, hint):
    print(("✅ " + ok_msg) if passed else ("❌ " + hint))

In [ ]:
%pip install -q -U "bitsandbytes>=0.46.1"
import bitsandbytes
print("bitsandbytes", bitsandbytes.__version__)

In [ ]:
import json, random

PROMPT = "Extract the person's name, age and city as JSON."
NAMES = ["Sara", "Omar", "Lina", "Youssef", "Mona", "Karim", "Nour", "Emma", "Liam", "Sofia",
         "Mateo", "Aiko", "Kenji", "Priya", "Arjun", "Chloe", "Lucas", "Fatima", "Diego", "Elena",
         "Leo", "Amara", "Kofi", "Hana", "Ines"]          # last 5 are test-only
CITIES = ["Cairo", "Alexandria", "Paris", "Lagos", "Tokyo", "Lima", "Berlin", "Toronto",
          "Mumbai", "Seoul", "Nairobi", "Madrid", "Dubai", "Oslo", "Austin", "Lisbon",
          "Hanoi", "Accra", "Quito", "Perth"]                # last 4 are test-only
TEMPLATES = [
    "Hi, I'm {name} from {city}. I'm {age}.",
    "{name} ({age}) lives in {city}.",
    "My name is {name}, I am {age} years old and I live in {city}.",
    "New signup: {name}, age {age}, based in {city}.",
    "Hello from {city}! {name} here, just turned {age}.",
    "{city}-based {name}, {age}, is joining the team next week.",
    "Contact: {name} | {age} | {city}",
    "Applicant {name} is {age} and currently resides in {city}.",
    "Meet {name}: {age} years young and loving life in {city}.",   # test-only
    "I moved to {city} last year. Name's {name}, age {age}.",      # test-only
]

def make_rows(n, split="train", seed=0):
    """Return n chat examples in TRL's prompt-completion format."""
    rng = random.Random(seed)
    test = split == "test"
    names = NAMES[20:] if test else NAMES[:20]
    cities = CITIES[16:] if test else CITIES[:16]
    templates = TEMPLATES[8:] if test else TEMPLATES[:8]
    rows = []
    for _ in range(n):
        person = {"name": rng.choice(names), "age": rng.randint(18, 80), "city": rng.choice(cities)}
        text = rng.choice(templates).format(**person)
        rows.append({
            "prompt": [{"role": "user", "content": f"{PROMPT}\n\nText: {text}"}],
            "completion": [{"role": "assistant", "content": json.dumps(person)}],
        })
    return rows

In [ ]:
def score(model, tokenizer, rows, show=3):
    """Greedy-generate a reply for each row; report valid-JSON rate and field accuracy."""
    model.eval()
    valid, fields_right = 0, 0
    for i, row in enumerate(rows):
        gold = json.loads(row["completion"][0]["content"])
        inputs = tokenizer.apply_chat_template(row["prompt"], add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
        reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            pred = json.loads(reply)
            valid += 1
        except json.JSONDecodeError:
            pred = {}
        if isinstance(pred, dict):
            fields_right += sum(pred.get(k) == v for k, v in gold.items())
        if i < show:
            print(f"  reply: {reply!r}")
    n = len(rows)
    result = {"valid_json": valid / n, "field_accuracy": fields_right / (3 * n)}
    print(result)
    return result

In [ ]:
from datasets import Dataset

train = Dataset.from_list(make_rows(300, "train", seed=0))
test_rows = make_rows(20, "test", seed=1)   # 20 rows: generation with a 1.7B model is slower
print(len(train), "train rows |", len(test_rows), "test rows")

## 1 · Predict the memory before you load anything
Use the bytes-per-parameter table from the lesson (section 1).

**TODO 1:** Estimate, in GB, (a) a full fine-tune of this model with AdamW and (b) just its weights stored in 4-bit.

In [ ]:
n_params = 1.7e9            # SmolLM2-1.7B

# TODO 1
full_ft_gb = ...
qlora_weights_gb = ...

In [ ]:
# check 1
ok_full = isinstance(full_ft_gb, float) and 25 < full_ft_gb < 35
ok_4bit = isinstance(qlora_weights_gb, float) and 0.6 < qlora_weights_gb < 1.1
check(ok_full, "Full fine-tune estimate is right: it does not fit a 16 GB T4.",
      "Multiply n_params by the bytes per parameter of a full fine-tune with AdamW (weights + gradients + optimizer), then divide by 1e9.")
check(ok_4bit, "4-bit weight estimate is right.",
      "How many bytes does one 4-bit number take? Use that per parameter, then convert to GB.")
if ok_full and ok_4bit:
    print(f"full fine-tune ≈ {full_ft_gb:.1f} GB | 4-bit weights ≈ {qlora_weights_gb:.2f} GB | a T4 has 16 GB")

## 2 · Describe the 4-bit format
In Transformers v5 you describe quantization with a `BitsAndBytesConfig` object. (Old tutorials pass `load_in_4bit=True` straight to `from_pretrained`. v5 removed that.)

**TODO 2:** Fill in the two blanks: NF4 as the 4-bit type, and `float16` as the compute dtype (the T4 has no bf16). Double quantization is already on.

In [ ]:
from transformers import BitsAndBytesConfig

# TODO 2
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=...,
    bnb_4bit_use_double_quant=True,     # also quantize the per-block scales
    bnb_4bit_compute_dtype=...,
)

In [ ]:
# check 2
check(bnb_config.bnb_4bit_quant_type == "nf4", "NF4 is set.",
      "bnb_4bit_quant_type takes a string: the 4-bit type from the QLoRA paper, in lower case.")
check(bnb_config.bnb_4bit_compute_dtype == torch.float16, "Compute dtype is float16.",
      "Pass a torch dtype object (torch.something), not a string. The T4 needs the 16-bit type that isn't bf16.")

## 3 · Load the model in 4-bit and measure it
`dtype=torch.float16` applies to the few layers that stay unquantized (embeddings, norms). A 4-bit model goes straight onto the GPU, so there's no `.to(device)`.

**TODO 3:** Pass your config to `from_pretrained`, then read the model's size in GB with `model.get_memory_footprint()` (it returns bytes).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)

# TODO 3
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, quantization_config=...)  # dtype: embeddings + norms
footprint_gb = ...

In [ ]:
# check 3
check(getattr(model, "is_loaded_in_4bit", False), "The model is loaded in 4-bit.",
      "Pass the BitsAndBytesConfig you built in TODO 2.")
check(isinstance(footprint_gb, float) and 0 < footprint_gb < 3, "Footprint measured in GB.",
      "The model can report its own memory footprint in bytes. Convert bytes to GB.")
if isinstance(footprint_gb, float):
    print(f"model in memory: {footprint_gb:.2f} GB")

How does the bigger base model do before any training? Look at the *style* of its replies, not just the numbers.

In [ ]:
print("BEFORE fine-tuning:")
before = score(model, tok, test_rows)

## 4 · Train LoRA adapters on the 4-bit model
The frozen 4-bit weights are never updated. Gradients flow through them into the LoRA adapters. The three memory knobs are spelled out in the code.

**TODO 4:** Set the two precision flags for a T4: no bf16, use fp16 mixed precision instead.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM")
args = SFTConfig(
    output_dir="qlora-out", num_train_epochs=1, learning_rate=2e-4,
    per_device_train_batch_size=4,      # memory knob 1: small batches...
    gradient_accumulation_steps=2,      # ...accumulated: 4 x 2 = effective batch of 8
    gradient_checkpointing=True,        # memory knob 2 (already SFTConfig's default)
    max_length=128,                     # memory knob 3: our longest example is 91 tokens
    bf16=..., fp16=...,                 # TODO 4 (fp16 = mixed-precision training)
    logging_steps=10, save_strategy="no", report_to="none",
)

In [ ]:
# check 4
check(args.fp16 is True and args.bf16 is False, "fp16 on, bf16 off: T4-safe.",
      "The T4 has no native bf16. Turn that flag off and turn on the other 16-bit format.")

In [ ]:
import time

trainer = SFTTrainer(model=model, args=args, train_dataset=train, processing_class=tok,
                     peft_config=peft_config)
# T4 fix: for 4-bit models TRL 1.15 casts the adapters to bf16, and fp16 training crashes
# on bf16 gradients. Upcast just the adapters (about 1% of the parameters) to fp32.
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
trainer.model.print_trainable_parameters()

torch.cuda.reset_peak_memory_stats()   # measure the training run only
t0 = time.time()
trainer.train()
print(f"train time: {time.time() - t0:.0f} s")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Compare with BEFORE: same numbers? Same reply style?

In [ ]:
print("AFTER QLoRA fine-tuning:")
after = score(trainer.model, tok, test_rows)
check(after["valid_json"] >= 0.9, "Held-out valid-JSON rate is 90% or more on a 1.7B model, on a free GPU.",
      "Lower than expected. Check TODO 4 and the training log, and re-run from section 3.")

## Stretch (optional)
Re-run from **section 3** for each experiment, so you start from a clean model.
- Compare the peak GPU memory above with your TODO 1 estimate for a full fine-tune. Where did the savings come from?
- Set `per_device_train_batch_size=1, gradient_accumulation_steps=8`. The effective batch is the same. What happens to peak memory and train time?
- Save the adapter with `trainer.model.save_pretrained("smollm2-1.7b-json-qlora")` and check its size, as in Lesson 6.